# Kinetic Profiles from Paired VEST Diagnostics

## Build shot 48224 at 300 ms

In [ ]:
from pathlib import Path
import os

OUTPUT_DIR = Path(os.environ.get("VAFT_DOCS_OUTPUT_DIR", "notebooks/outputs/docs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib.pyplot as plt
import numpy as np
import vaft
from vaft.code.efit import build_kinetic_core_profiles
from vaft.data import read_geqdsk

shot = 48224
time_ms = 300.0
data_root = Path("vaft/data/kineticEfit")
geq = read_geqdsk(data_root / "g048224.00300")
ods = geq.to_omas()
vaft.machine_mapping.dataset_description(
    ods, source=shot,
    options={"source_type": "shot", "description": "Paired documentation sample"},
)
vaft.machine_mapping.thomson_scattering(ods, shot, data_root / "NeTe_48224.mat")
vaft.machine_mapping.charge_exchange(
    ods, shotnumber=shot, options="ids", mat_file=data_root / "IDS_48224.mat"
)
ods = build_kinetic_core_profiles(
    ods, geq, time_ms,
    te_mode="polynomial", ne_mode="polynomial",
    ti_mode="polynomial", vtor_mode="polynomial",
)

profile = ods["core_profiles.profiles_1d.0"]
rho = np.asarray(profile["grid.rho_tor_norm"], dtype=float)
te = np.asarray(profile["electrons.temperature"], dtype=float)
ne = np.asarray(profile["electrons.density"], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].plot(rho, te, color="tab:red", lw=2)
axes[0].set(xlabel=r"$\rho_{tor,norm}$", ylabel=r"$T_e$ [eV]", title="Electron temperature")
axes[1].plot(rho, ne / 1e19, color="tab:blue", lw=2)
axes[1].set(xlabel=r"$\rho_{tor,norm}$", ylabel=r"$n_e$ [$10^{19}$ m$^{-3}$]", title="Electron density")
for ax in axes:
    ax.grid(alpha=0.25)
fig.suptitle("VEST shot 48224 at 300 ms")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "kinetic-profile.png", dpi=180, bbox_inches="tight")
plt.show()
print({"shot": shot, "time_ms": time_ms, "grid_points": int(rho.size)})
